# 🚀 ZONARETH-LM: GPU Training Studio (Google Colab)
Welcome to the official training studio for **ZONARETH-LM**!

This notebook trains your custom language model end-to-end on GPU:
1. **GPU & Environment Check**
2. **Codebase Setup & Dependencies**
3. **Dataset Ingestion** (Auto-download TinyStories/FineWeb, or custom dataset)
4. **BPE Tokenizer & Preprocessing**
5. **Transformer GPU Training** (with cosine LR, AMP float16, checkpointing)
6. **Interactive Model Testing** (Generate text directly in the notebook)
7. **Download Artifacts** (.zip archive of weights, tokenizer, config)

In [ ]:
# Step 1: Verify GPU Accelerator
import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detected: {device_name} ({vram_gb:.2f} GB VRAM)")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ No GPU detected!")
    print("   On Colab: Go to Runtime -> Change runtime type -> Select 'T4 GPU'.")

In [ ]:
# Step 2: Clone Repository & Install Dependencies
import os, sys
from pathlib import Path

if not Path("Veyra").exists() and not Path("Zonareth").exists() and not Path("configs").exists():
    !git clone https://github.com/shreyashmane-dev/Veyra.git
    %cd Veyra
elif Path("Veyra").exists() and not Path("configs").exists():
    %cd Veyra
elif Path("Zonareth").exists() and not Path("configs").exists():
    %cd Zonareth

!pip install -e . -q
!pip install datasets tqdm pyyaml -q
print("\n✅ ZONARETH codebase and dependencies ready!")

### 📂 Step 3: Dataset Ingestion & Preparation
You can train ZONARETH using:
- **Option A (Automatic - Zero friction)**: Automated download of TinyStories or FineWeb-Edu.
- **Option B (Custom Upload)**: Drop any `.txt` or `.jsonl` file into `data/raw/`.

In [ ]:
# Step 3: Ingest or Download Training Data
from pathlib import Path

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# Ensure at least one dataset is ready
raw_files = list(raw_dir.glob("*.txt")) + list(raw_dir.glob("*.jsonl"))
if not raw_files:
    print("Downloading TinyStories pretraining corpus (50 MB)...")
    !python scripts/download_pretraining_data.py --dataset tinystories --max-mb 50
    raw_files = list(raw_dir.glob("*.txt")) + list(raw_dir.glob("*.jsonl"))

print(f"✅ Training data ready ({len(raw_files)} files):")
for f in raw_files:
    print(f"   - {f.name} ({f.stat().st_size / (1024*1024):.2f} MB)")

### ⚡ Step 4: Launch GPU Training Pipeline
The pipeline will:
1. Train a custom Byte-Pair Encoding (BPE) tokenizer (`ZonarethTokenizer`).
2. Preprocess and shard tokens into binary tensors.
3. Instantiate the **ZONARETH** Transformer architecture (Rotary embeddings, RMSNorm, SwiGLU, Multi-Query Attention).
4. Execute the training loop with FP16/BF16 mixed precision, cosine learning rate schedule, and periodic checkpoints.

> **Recommended settings:**
> - For full LLM: `--model 125m --steps 2500 --batch-size 8`
> - For quick test: `--model tiny --steps 300 --batch-size 4`

In [ ]:
# Step 4: Run Training Pipeline
!python scripts/run_kaggle_pipeline.py --model 125m --steps 2500 --batch-size 8 --save-interval 250 --max-to-keep 2


### 💬 Step 5: Test Your Trained Model!
Let's load the trained weights and generate text directly inside this notebook.

In [ ]:
# Step 5: Inference with Trained ZONARETH-LM
import torch
from pathlib import Path

try:
    from zonareth.inference.engine import ZonarethInferenceEngine
except ImportError:
    from veyra.inference.engine import VeyraInferenceEngine as ZonarethInferenceEngine

ckpt_candidates = [
    Path("/kaggle/working/zonareth_125m/latest.pt"),
    Path("/kaggle/working/zonareth_125m/best.pt"),
    Path("checkpoints/zonareth_125m/latest.pt"),
    Path("checkpoints/zonareth_tiny/latest.pt"),
]

ckpt_file = next((p for p in ckpt_candidates if p.exists()), None)
tok_path = Path("data/tokenizer")

if ckpt_file and tok_path.exists():
    print(f"✅ Loading checkpoint: {ckpt_file}")
    engine = ZonarethInferenceEngine.from_checkpoint(str(ckpt_file), str(tok_path))
    
    test_prompts = [
        "Once upon a time",
        "The brave little robot",
        "In a secret hidden forest",
    ]
    
    print("\n" + "=" * 60)
    print("   ZONARETH-LM TEXT GENERATION SAMPLES")
    print("=" * 60)
    for prompt in test_prompts:
        print(f"\nPrompt : {prompt}")
        output = engine.generate(prompt, max_new_tokens=60, temperature=0.8, top_k=40)
        print(f"ZONARETH: {output}")
else:
    print("⚠️ Checkpoint not found. Run Step 4 first!")

### 📦 Step 6: Download Trained Artifacts
Download your model checkpoint, tokenizer, and config archive (`.zip`) directly to your local computer.

In [ ]:
# Step 6: Verify and Download Artifacts Archive
from google.colab import files
from pathlib import Path

working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("checkpoints")
archives = list(working_dir.glob("*.zip")) + list(Path(".").glob("*.zip"))

if archives:
    print(f"✅ Downloading {archives[0].name}...")
    files.download(str(archives[0]))
else:
    print("No .zip archives found yet.")